# **Similarity Measures for RAG Pipeline** 

### 1. Semantic Similarities (Sentence-BERT Cross-Encoder)

In [ ]:
from sentence_transformers import CrossEncoder

def setup_cross_encoder():
    """More accurate but slower semantic similarity"""
    return CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def cross_encoder_similarity(cross_encoder, user_prompt, stored_prompts):
    """Get semantic similarity using cross-encoder"""
    pairs = [[user_prompt, stored_prompt] for stored_prompt in stored_prompts]
    scores = cross_encoder.predict(pairs)
    return scores.tolist()

# Usage in your pipeline
cross_encoder = setup_cross_encoder()

def enhanced_similarity_search(collection, embedder, user_prompt: str, top_k: int = 8):
    # First get candidates with cosine similarity
    initial_results = query_similar_prompts(collection, embedder, user_prompt, top_k*2)
    
    # Re-rank with cross-encoder
    stored_prompts = [hit["adversarial_prompt"] for hit in initial_results]
    cross_encoder_scores = cross_encoder_similarity(cross_encoder, user_prompt, stored_prompts)
    
    # Update similarity scores
    for hit, new_score in zip(initial_results, cross_encoder_scores):
        hit["similarity"] = float(new_score)
    
    return sorted(initial_results, key=lambda x: x["similarity"], reverse=True)[:top_k]

### 2. Universal Sentence Encoder 

In [ ]:
import tensorflow_hub as hub

def setup_use():
    """Load Universal Sentence Encoder"""
    return hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

def use_similarity_search(use_model, user_prompt, stored_prompts):
    """Similarity using Universal Sentence Encoder"""
    all_texts = [user_prompt] + stored_prompts
    embeddings = use_model(all_texts).numpy()
    
    user_embedding = embeddings[0]
    stored_embeddings = embeddings[1:]
    
    similarities = []
    for stored_emb in stored_embeddings:
        sim = np.dot(user_embedding, stored_emb) / (np.linalg.norm(user_embedding) * np.linalg.norm(stored_emb))
        similarities.append(float(sim))
    
    return similarities

### 3. Bert based similarity 



In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

class BERTSimilarity:
    def __init__(self, model_name='bert-base-uncased'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
    
    def get_embeddings(self, texts):
        """Get BERT embeddings"""
        encoded = self.tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
        
        with torch.no_grad():
            outputs = self.model(**encoded)
            # Use CLS token embedding
            embeddings = outputs.last_hidden_state[:, 0, :]
        
        return embeddings
    
    def similarity_search(self, user_prompt, stored_prompts, top_k=8):
        """BERT-based similarity search"""
        all_texts = [user_prompt] + stored_prompts
        embeddings = self.get_embeddings(all_texts)
        
        user_emb = embeddings[0].unsqueeze(0)
        stored_embs = embeddings[1:]
        
        # Cosine similarity
        similarities = F.cosine_similarity(user_emb, stored_embs, dim=1)
        
        # Get top-k
        top_indices = torch.argsort(similarities, descending=True)[:top_k]
        
        results = []
        for idx in top_indices:
            results.append({
                'similarity': float(similarities[idx]),
                'document': stored_prompts[idx],
                'index': int(idx)
            })
        
        return results

# **Potential Datasets to Swap with I2P**

# Violence Related Datasets

### 1. `ross-dev/violence-dataset`
- **Type:** Video dataset  
- **Contents:** Short real-world video clips labeled as either `Violence` or `NonViolence`.  
- **Purpose:** Designed for binary classification of violent vs. non-violent activity in surveillance and real-life footage.  
- **Use Case for Our Project:** While the dataset itself is video, the **labels** and any available metadata/captions can be adapted into *prompt-style text* for our RAG store.  
  - Example: “two people fighting in the street” → labeled as `Violence`.

---

### 2. `valiantlynxz/godseye-violence-detection-dataset`
- **Type:** Composite dataset (multi-source)  
- **Contents:** Aggregates several popular violence detection datasets, including:  
  - **RWF-2000** – Real-world fight videos.  
  - **Hockey Fight Dataset** – Clips of violent vs non-violent scenes in hockey matches.  
  - **Airtlab Violence Dataset** – Additional real-life violent scenarios.  
- **Purpose:** Provides a **diverse, large-scale set** of violent vs non-violent samples for training and evaluation.  
- **Use Case for Our Project:** Because it combines multiple sources, it gives a **variety of violent contexts** (sports fights, street fights, real-world CCTV).  
  - Violent sample descriptions can be extracted to improve coverage in our vector DB.

---

### 3. `jherng/xd-violence`
- **Type:** Large-scale video dataset  
- **Contents:** Videos labeled for **violence detection** across multiple contexts.  
- **Highlights:**  
  - Broader range than RWF/Hockey/Airtlab individually.  
  - Includes crowded scenarios, weapon use, and fight sequences.  
- **Purpose:** Benchmark dataset for violence detection in videos, especially for **multi-modal ML tasks**.  
- **Use Case for Our Project:** We can generate or extract **prompt-like descriptions** of violent activities (e.g., “a man attacking another with a weapon”), which align closely with what our RAG filter needs.

---

 **Summary:**  
- `ross-dev/violence-dataset` → focused, real-world binary classification (Violence vs NonViolence).  
- `godseye-violence-detection-dataset` → composite, diverse, multi-source dataset (sports, street, real-life).  
- `xd-violence` → large-scale, multi-context dataset with broad coverage of violent activities.


In [ ]:
# ross-dev / violence-dataset (video-based, but metadata + labels available)
!huggingface-cli download ross-dev/violence-dataset --repo-type dataset --local-dir ./violence_dataset

# valiantlynxz / godseye-violence-detection-dataset (multi-source violence data)
!huggingface-cli download valiantlynxz/godseye-violence-detection-dataset --repo-type dataset --local-dir ./godseye_violence

# jherng / xd-violence (large-scale violence dataset)
!huggingface-cli download jherng/xd-violence --repo-type dataset --local-dir ./xd_violence
